# 제품 트레이스와 자율성 진단

공식 저장소가 공개한 실제 실행 궤적 두 건과 제품 추적 단서를 읽고 중간 프로젝트 진단표를 완성합니다.

## 실행 환경 준비

`public-trace-extracts.json`은 공개된 SWE-agent·mini-SWE-agent 원시 궤적에서 사건 순서를 추출한 자료입니다. 원본 커밋·URL·SHA-256·MIT 라이선스를 함께 기록했습니다. 이 컴퓨터에서 두 제품을 다시 실행한 로그는 아닙니다.

In [1]:
# 이 셀의 계산과 출력에서 현재 단계의 결과를 확인합니다.
from pathlib import Path
import json, sys

start = Path.cwd().resolve()
candidates = (start, *start.parents)
ROOT = next(path for path in candidates if (path / "labs/part5/ch17-agent-overview").is_dir())
LAB = ROOT / "labs/part5/ch17-agent-overview"
print("Python:", sys.version.split()[0], "| 실습 폴더:", LAB.relative_to(ROOT))

Python: 3.12.13 | 실습 폴더: labs/part5/ch17-agent-overview


## 실제 공개 궤적 읽기

첫 궤적은 14개의 action-observation 턴, 둘째 궤적은 10개의 명령과 반환 결과를 포함합니다.

In [2]:
# 이 셀의 계산과 출력에서 현재 단계의 결과를 확인합니다.
traces = json.loads((LAB / "data/public-trace-extracts.json").read_text())
for trace in traces:
    print(trace["product"], "| 사건:", len(trace["events"]), "| SHA-256:", trace["sha256"][:12])
    print(" 경로:", " → ".join(event["action"] for event in trace["events"]))
    print(" 출처:", trace["raw_url"])
assert [len(trace["events"]) for trace in traces] == [14, 10]
assert all(len(trace["sha256"]) == 64 for trace in traces)

SWE-agent v0.7.0 demonstration | 사건: 14 | SHA-256: 8856076ec318
 경로: ls → open setup.py → install dev dependencies → create reproduction → write reproduction → run reproduction → list files → find fields.py → open target lines → edit without indentation → edit with indentation → rerun reproduction → remove reproduction → submit patch
 출처: https://raw.githubusercontent.com/SWE-agent/SWE-agent/3ea751c087f32b16e039a2233dd6eefecef325d5/trajectories/demonstrations/replay__marshmallow-code__marshmallow-1867__default__t-0.20__p-0.95__c-2.00__install-1___install_from_source/marshmallow-code__marshmallow-1867.traj
mini-SWE-agent GitHub issue fixture | 사건: 10 | SHA-256: 86d060a4881e
 경로: read absolute path → list repository → list tests → read target file → add missing colon → read edited file → run script → run issue input → add zero check → submit plus git diff
 출처: https://raw.githubusercontent.com/SWE-agent/mini-swe-agent/04d809ceab9df28f9adaed044884180159172930/tests/test_data/github_issue.

## 오류 뒤 경로 변경 확인

원시 궤적에서 반환 실패나 편집 거부 다음에 실제 행동이 바뀐 위치를 읽습니다.

In [3]:
# 이 셀의 계산과 출력에서 현재 단계의 결과를 확인합니다.
for trace in traces:
    failures = [event for event in trace["events"] if event["outcome"] == "failure"]
    print(trace["product"], "실패 사건:", [(e["step"], e["action"], e["evidence"]) for e in failures])
    print(" 관찰 뒤 다음 행동:", trace["recovery_pairs"])
    print(" 하드코딩 경계:", trace["hardcoded_boundary"])
assert traces[0]["events"][9]["outcome"] == "failure"
assert traces[1]["events"][-1]["outcome"] == "unconfirmed"

SWE-agent v0.7.0 demonstration 실패 사건: [(10, 'edit without indentation', 'E999 IndentationError; edit rejected')]
 관찰 뒤 다음 행동: [[10, 11, '편집 거부 뒤 들여쓰기를 고쳐 다시 편집']]
 하드코딩 경계: 도구 문법 검사기가 들여쓰기 오류가 있는 편집을 거부하고 submit 도구가 종료를 표시합니다.
mini-SWE-agent GitHub issue fixture 실패 사건: [(1, 'read absolute path', 'returncode 1; file not found'), (8, 'run issue input', 'returncode 1; ZeroDivisionError')]
 관찰 뒤 다음 행동: [[1, 2, '잘못된 절대 경로 뒤 현재 저장소를 목록으로 확인'], [8, 9, 'ZeroDivisionError 뒤 0 검사 추가']]
 하드코딩 경계: 시스템 지시는 응답당 명령 하나와 단독 완료 명령을 요구하지만 마지막 행동은 완료 명령에 git 명령을 결합했습니다.


## 공식 제품 추적 단서와 증거 수준

Codex·Claude Code·Deep Research 문서는 추적 인터페이스와 행동 유형을 확인하는 보조 자료입니다. 실제 단일 실행 원문과 구분합니다.

In [4]:
# 이 셀의 계산과 출력에서 현재 단계의 결과를 확인합니다.
products = json.loads((LAB / "data/product-trace-evidence.json").read_text())
for product in products:
    print(product["product"], product["artifact_type"], "| 로컬 실행:", product["local_execution"])
    print(" 한계:", product["limitation"])
assert all(product["local_execution"] is False for product in products)

OpenAI Codex official_public_trace_signals | 로컬 실행: False
 한계: 공식 소개가 공개한 증거 유형이며 하나의 전체 실행 원문은 공개되지 않았습니다.
Anthropic Claude Code official_public_trace_interface | 로컬 실행: False
 한계: CLI가 제공하는 추적 인터페이스이며 특정 사용자 작업의 실제 출력은 포함하지 않습니다.
OpenAI Deep Research official_public_behavior_trace | 로컬 실행: False
 한계: 공식 설명의 행동 경로를 사건으로 추출했으며 원시 내부 추론이나 완전한 실행 로그가 아닙니다.


## 작성된 진단 예시

중간 프로젝트 RAG 예시는 검색과 답변 경로가 정해져 있어 워크플로우 유지로 판정했습니다.

In [5]:
# 이 셀의 계산과 출력에서 현재 단계의 결과를 확인합니다.
import csv

example = json.loads((LAB / "data/autonomy-diagnostic-example.json").read_text())
for key, value in example.items():
    print(f"{key}: {value}")
sheet = list(csv.DictReader((LAB / "data/autonomy-diagnostic-sheet.csv").open()))
print("재사용 진단 질문 수:", len(sheet))
assert example["recommended_architecture"] == "워크플로우 유지"

project: 중간 프로젝트 규정 RAG
goal: 승인된 규정 문서에서 답과 출처를 반환
path_predefined: yes
unknown_steps_required: no
model_decides_next_action: no
write_side_effect: no
stop_conditions: 검색 결과 없음, 출처 검증 실패, 답변 반환
evidence: retrieve → generate → citation_validate 순서가 모든 질문에 동일
recommended_architecture: 워크플로우 유지
reason: 경로가 고정되어 있으며 자율적 재검색의 품질 이득을 아직 측정하지 않았습니다.
next_measurement: Ch 20에서 고정 분기와 모델 분기의 호출 수·비용·결과 편차 비교
재사용 진단 질문 수: 9
